In [1]:
from cuTen import cuten
import numpy as np
from Seera_init import tensor as Tensor
import matplotlib.pyplot as plt
from Seera import (
    Input, Conv2D, MaxPool2D, Flatten, Dense,
    Sequential, Loss, Adam,
)
from Seera_Engine import autograd4nn
na = np.random.randn(3,2,2)
nb = (np.arange(0,4,dtype = np.float32)+10).reshape(2,2)


a = cuten(na,dtype="float32")
b = cuten(nb,dtype = "float32")


In [2]:
a

[[[ 1.3166912  -1.0577797 ]
  [-0.49626356 -0.6427788 ]]

 [[ 0.0382379  -0.70583713]
  [ 0.5120746   0.45385078]]

 [[-0.4689542   0.19288556]
  [-0.34603268 -1.809316  ]]]


IT DOES WORK

In [3]:
b


[[10. 11.]
 [12. 13.]]


IT DOES WORK

In [4]:
a@b

[[[  0.47265625   0.7314453 ]
  [-12.674316   -13.813232  ]]

 [[ -8.090271    -8.758087  ]
  [ 10.568359    11.534424  ]]

 [[ -2.3754883   -2.6516113 ]
  [-25.174316   -27.329834  ]]]


IT DOES WORK

In [5]:
A = np.array([[1, 2], [3, 4]], dtype=np.float32)
B = np.eye(2)
expected = A @ B  # [[19,22],[43,50]]
result = (cuten(A).matmul(cuten(B)))

In [6]:
expected

array([[1., 2.],
       [3., 4.]])

In [7]:
result

[[1. 2.]
 [3. 4.]]


IT DOES WORK

In [8]:
result.to_host_f32()

array([[1., 2.],
       [3., 4.]], dtype=float32)

In [9]:

from tensorflow.keras.datasets import mnist
import numpy as np
(X_train, y_train), (X_test, y_test) = mnist.load_data()

X_train_ = np.expand_dims(X_train,axis=1)/255
y_train_ = np.zeros((60000,10))
for i in range (0,60000):
    y_train_[i,:] = np.eye(1,10,y_train[i])

2026-04-09 18:13:07.032862: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-09 18:13:07.080007: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-09 18:13:08.003653: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


KeyboardInterrupt: 

In [ ]:
y_train_ = y_train_[:,:,np.newaxis]

In [ ]:
y_train_.shape

(60000, 10, 1)

In [ ]:
X_test_ = np.expand_dims(X_test,axis=1)/255
print("=" * 60)
print("  MNIST GPU Test — Seera Framework (cuda)")
print("=" * 60)

print(f"Train: {X_train.shape}, Test: {y_test.shape}")

# ─── Build Model (device="cuda") ─────────────────────────
model = Sequential([
    Input((1, 28, 28)),
    # Conv2D(8, 1, (3, 3), activation="relu", stride=1, zero_padding=1),
    # MaxPool2D(pool_size=(2, 2), stride=2),
    # Conv2D(16, 8, (3, 3), activation="relu", stride=1, zero_padding=1),
    # MaxPool2D(pool_size=(2, 2), stride=2),
    Flatten(),
    Dense(28*28, 2*64, activation="relu"),
    Dense(2*64, 64, activation="relu"),
    
    Dense(64, 10, activation="softmax"),
], device="cuda")
model.summary()

# ─── Train ───────────────────────────────────────────────
loss_fn = Loss()
optimizer = Adam(model, lr=0.001)


idx = np.random.permutation(len(X_train_))
X, y = X_train_[idx], y_train_[idx]  # shuffle (assuming labels y)

batch_size = 4

loss_track = 0.0
for epoch in range(5):
    epoch_loss = 0.0
    n_batches = 0
    for i in range(0, len(X), batch_size):
        X_batch, y_batch = X[i:i+batch_size], y[i:i+batch_size]
        X_batch = Tensor(X_batch, is_leaf=True, device="cuda")
        y_batch = Tensor(y_batch, device="cuda")
        pred = model.forward(X_batch)
        loss = loss_fn.categorical_cross_entropy(pred, y_batch)
        loss_val = float(loss.value.to_host_f32().ravel()[0])
        epoch_loss += loss_val
        n_batches += 1

        model.zero_grad()
        autograd4nn(loss)
        optimizer.step()
    avg_loss = epoch_loss / n_batches
    print(f"EPOCH {epoch}/5: Loss: {avg_loss:.6f}")
# history = model.fit(
#     X_train_, y_train_,
#     Optimizer=optimizer,
#     Loss=loss_fn.categorical_cross_entropy,
#     Epochs=5,
#     batch_size=16,
#     Loss_interval=1,
# )

# ─── Evaluate ────────────────────────────────────────────
correct = 0
for i in range(len(X_test)):
    x = Tensor(X_test_[i:i+1], is_leaf=True, device="cuda")
    pred = model.predict(x)
    # bring to host for argmax
    pred_np = pred.value.to_host_f32()
    pred_label = np.argmax(pred_np)
    if pred_label == y_test[i]:
        correct += 1

accuracy = correct / len(X_test) * 100
print(f"\nTest Accuracy: {accuracy:.1f}% ({correct}/{len(X_test)})")
print("GPU test complete ✓")

  MNIST GPU Test — Seera Framework (cuda)
Train: (60000, 28, 28), Test: (10000,)
Model Summary
  Layer 0: Input Layer with shape (1, 28, 28)
  Layer 1: Flatten Layer
  Layer 2: Dense(784→128, act=relu, params=100480)
  Layer 3: Dense(128→64, act=relu, params=8256)
  Layer 4: Dense(64→10, act=softmax, params=650)
EPOCH 0/5: Loss: 0.014325
EPOCH 1/5: Loss: 0.000000


KeyboardInterrupt: 

In [ ]:
print("\n[10.3] Identity matmul")
A = np.random.randn(5, 5).astype(np.float32)
I = np.eye(5, dtype=np.float32)
result =(cuten(A).matmul(cuten(I))).to_host_f32()



[10.3] Identity matmul


In [ ]:
A

array([[-0.4456001 , -1.0393741 ,  0.99846375,  0.22814934,  0.97008723],
       [ 1.4876453 ,  0.5295499 ,  1.5039212 ,  0.21904601,  0.03024955],
       [-0.2982236 , -0.24424252,  2.0417113 , -0.82809716, -0.21027498],
       [ 0.69669795,  0.56495357, -1.0144784 ,  0.75709236, -1.233691  ],
       [ 0.83846897,  0.67054826, -1.0540261 ,  0.8622056 ,  2.6460516 ]],
      dtype=float32)

In [ ]:
(1,)+ (2,3,4,6)

(1, 2, 3, 4, 6)

In [9]:
from cuTen import cuten
import numpy as np

na = np.arange(2,18,dtype=np.float32).reshape(1,1,4,4)
# nb = np.eye(2,dtype=np.float32)
nb = np.float32(np.random.random((4,4)))
nb = nb[np.newaxis,np.newaxis,:,:]

X= cuten(na)

W = cuten(nb)

In [10]:
W

[[[[0.99602884 0.881166   0.43100822 0.6835572 ]
   [0.06246791 0.41641754 0.9733815  0.6458324 ]
   [0.9126385  0.8048417  0.494467   0.12426476]
   [0.33558583 0.90158176 0.07192528 0.4089498 ]]]]


IT DOES WORK

In [20]:
import torch

tX = torch.tensor(na)
tW = torch.tensor(nb)

torch.conv_transpose2d(tX,tW,stride=(2,2))

tensor([[[[ 1.9921,  1.7623,  3.8501,  4.0106,  5.2771,  5.5753,  6.7042,
            7.1401,  2.1550,  3.4178],
          [ 0.1249,  0.8328,  2.1342,  2.5409,  3.1700,  3.6032,  4.2059,
            4.6654,  4.8669,  3.2292],
          [ 7.8014,  6.8967, 13.2851, 12.9326, 16.1192, 15.4264, 18.9534,
           17.9202,  6.3514,  6.7733],
          [ 1.0460,  4.3017,  7.4282, 10.3126,  8.8715, 12.6853, 10.3149,
           15.0581,  9.1201,  7.8572],
          [15.4361, 13.6407, 24.6217, 22.9079, 27.4558, 25.4017, 30.2900,
           27.8955, 10.0533, 10.0046],
          [ 2.6382,  9.5737, 13.2016, 19.8037, 14.6450, 22.1765, 16.0883,
           24.5493, 13.3013, 12.0764],
          [23.0708, 20.3847, 35.9582, 32.8832, 38.7924, 35.3770, 41.6265,
           37.8709, 13.7552, 13.2359],
          [ 4.2304, 14.8457, 18.9751, 29.2948, 20.4184, 31.6676, 21.8618,
           34.0404, 17.4825, 16.2955],
          [12.7769, 11.2678, 20.6121, 13.8123, 22.0192, 14.7414, 23.4263,
           15.6705,  8

In [12]:
X.conv2d_transpose(W,strideh=2,stridew=2)

[[[[ 1.1025391  1.9931641  1.6538086  2.989746   2.2050781  3.9863281
     2.7563477  4.98291  ]
   [ 1.8496094  0.7910156  2.774414   1.1865234  3.6992188  1.5820312
     4.6240234  1.9775391]
   [ 3.3076172  5.979492   3.8588867  6.976074   4.4101562  7.9726562
     4.961426   8.969238 ]
   [ 5.548828   2.3730469  6.473633   2.7685547  7.3984375  3.1640625
     8.323242   3.5595703]
   [ 5.5126953  9.96582    6.063965  10.962402   6.6152344 11.958984
     7.166504  12.955566 ]
   [ 9.248047   3.9550781 10.172852   4.350586  11.097656   4.7460938
    12.022461   5.1416016]
   [ 7.7177734 13.952148   8.269043  14.94873    8.8203125 15.9453125
     9.371582  16.941895 ]
   [12.947266   5.5371094 13.87207    5.932617  14.796875   6.328125
    15.72168    6.723633 ]]]]


IT DOES WORK

In [15]:
out,mask = W.maxpool2d(2,2,2,2,1,1)

In [16]:
W


[[[[0.99602884 0.881166   0.43100822 0.6835572 ]
   [0.06246791 0.41641754 0.9733815  0.6458324 ]
   [0.9126385  0.8048417  0.494467   0.12426476]
   [0.33558583 0.90158176 0.07192528 0.4089498 ]]]]


IT DOES WORK

In [17]:
out

[[[[0.99602884 0.881166   0.6835572 ]
   [0.9126385  0.9733815  0.6458324 ]
   [0.33558583 0.90158176 0.4089498 ]]]]


IT DOES WORK

In [18]:
mask

[[[[1 1 0 1]
   [0 0 1 1]
   [1 0 0 0]
   [1 1 0 1]]]]


IT DOES WORK

In [28]:
pool = torch.nn.MaxPool2d((2,2),stride=(2,2),padding=(1,1),return_indices=True)

In [29]:
out,mask = pool(tW)

In [30]:
tW

tensor([[[[0.9960, 0.8812, 0.4310, 0.6836],
          [0.0625, 0.4164, 0.9734, 0.6458],
          [0.9126, 0.8048, 0.4945, 0.1243],
          [0.3356, 0.9016, 0.0719, 0.4089]]]])

In [31]:
out

tensor([[[[0.9960, 0.8812, 0.6836],
          [0.9126, 0.9734, 0.6458],
          [0.3356, 0.9016, 0.4089]]]])

In [33]:
mask

tensor([[[[ 0,  1,  3],
          [ 8,  6,  7],
          [12, 13, 15]]]])